# EV Charging Data Cleaning and Preparation

The cleaning process focuses on ensuring that:
- Each charging session has valid identifying information.
- Charging sessions can be connected to the correct vehicle.
- Charging sessions can be connected to the correct home or public charging station.
- Energy measurements and timestamps are in usable formats.
- Home and public charging sessions can be analyzed separately.

This preparation supports the project's three research questions:

1. How did weather conditions relate to electrical demand and carbon emissions?
2. How did at-home charging at night affect load on the power grid and carbon emissions?
3. How many emissions were generated from EVs charging across the city?

This notebook currently prepares the charging, vehicle, and station datasets. 

# 1. Import Libraries and Load the Datasets

Pandas and NumPy are used to load, inspect, clean, and integrate the datasets.

Three datasets are loaded for this stage:

- **Charging Sessions:** Contains individual EV charging sessions, including the vehicle, station, start time, end time, charging type, and energy delivered.
- **Vehicles:** Contains information about the 10,000 EVs, including vehicle identifiers, battery capacity, registration month, and gasoline vehicle information.
- **Public Stations:** Contains information about public charging stations and their identifiers.

Identifiers are loaded as strings to preserve their original formatting. 

In [1]:
import pandas as pd
import numpy as np

# Load datasets while preserving IDs as text
charging = pd.read_csv("charging_sessions.csv.gz", dtype={
    "session_id": "string",
    "vehicle_id": "string",
    "station_id": "string"
})

vehicles = pd.read_csv("vehicles.csv", dtype={
    "vehicle_id": "string",
    "home_zip": "string"
})

stations = pd.read_csv("stations.csv", dtype={
    "station_id": "string",
    "zip": "string"
})

print("Datasets loaded successfully!")

Datasets loaded successfully!


# 2. Record Original Row Counts

Before making any changes, the original number of rows in each dataset is recorded.

These counts provide a baseline for measuring how many records remain after cleaning.

The original and final row counts will be compared at the end of the notebook. Each cleaning operation is also recorded separately so that removed records and the reasons for their removal can be reported.

In [2]:
original_counts = {
    "Charging Sessions": len(charging),
    "Vehicles": len(vehicles),
    "Public Stations": len(stations)
}

for name, count in original_counts.items():
    print(f"{name}: {count:,} rows")

Charging Sessions: 2,927,187 rows
Vehicles: 10,000 rows
Public Stations: 239 rows


# 3. Inspect the Original Datasets

The inspection reports:
- Dataset dimensions (rows and columns).
- Missing values in each column.
- Completely duplicated rows.
- Data types for each column.

This initial inspection helps identify problems that may affect later analysis, including missing energy measurements, duplicate records, incorrect data types, and incomplete identifiers.

The results are used to determine which cleaning steps are necessary.

In [3]:
for name, df in {
    "Charging Sessions": charging,
    "Vehicles": vehicles,
    "Public Stations": stations
}.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("Shape:", df.shape)

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nDuplicate rows:", df.duplicated().sum())

    print("\nData types:")
    print(df.dtypes)


Charging Sessions
Shape: (2927187, 9)

Missing values:
session_id            0
vehicle_id            0
station_id            0
port_type             0
connector             0
start_time            0
end_time              0
kwh               44045
home_or_public        0
dtype: int64

Duplicate rows: 14563

Data types:
session_id         string
vehicle_id         string
station_id         string
port_type             str
connector             str
start_time            str
end_time              str
kwh               float64
home_or_public        str
dtype: object

Vehicles
Shape: (10000, 10)

Missing values:
vehicle_id              0
make                    0
model                   0
trim                    0
model_year              0
battery_kwh             0
rated_kwh_per_100mi     0
registration_month      0
replaced_gas_car_mpg    0
home_zip                0
dtype: int64

Duplicate rows: 0

Data types:
vehicle_id               string
make                        str
model           

# 4. Create Cleaning and Data-Quality Logs

Two logs are created to document the cleaning process.

**Cleaning log:** Records operations that remove rows. For each operation, it records:
- Dataset name.
- Cleaning operation.
- Number of rows before cleaning.
- Number of rows after cleaning.
- Number of rows removed.
- Reason for the operation.

**Data-quality log:** Records invalid values or conversions that do not necessarily remove entire rows. For example, a value that cannot be converted into a number is changed to a missing value (`NaN`) and recorded.

Keeping these logs separate makes it possible to distinguish between records removed from the analysis and individual values that require further investigation.

In [4]:
cleaning_log = []

def log_cleaning(dataset, step, before, after, reason):
    cleaning_log.append({
        "Dataset": dataset,
        "Cleaning Step": step,
        "Rows Before": before,
        "Rows After": after,
        "Rows Removed": before - after,
        "Reason": reason
    })

def show_cleaning_log():
    return pd.DataFrame(cleaning_log)

In [5]:
# Track data-quality issues that do not necessarily remove rows
quality_log = []

def log_quality(dataset, column, affected, action, reason):
    quality_log.append({
        "Dataset": dataset,
        "Column": column,
        "Affected Values": affected,
        "Action": action,
        "Reason": reason
    })

def show_quality_log():
    return pd.DataFrame(quality_log)

# 5. Clean the Vehicle Dataset

The vehicle dataset is cleaned first because vehicle identifiers are needed to verify and connect charging sessions.

The following operations are performed:

1. **Standardize vehicle IDs:** Remove extra spaces and convert IDs to uppercase so that formatting differences do not prevent matches.
2. **Remove missing vehicle IDs:** Records without an ID cannot be reliably connected to charging sessions.
3. **Remove duplicate vehicle IDs:** Each vehicle should appear only once in the vehicle dataset. When duplicate IDs occur, the first record is retained.
4. **Convert numerical columns:** Battery capacity, rated energy consumption, and replaced gasoline vehicle MPG are converted to numeric values. Values that cannot be converted become `NaN` and are recorded in the data-quality log.
5. **Convert registration month:** Registration dates are converted into datetime values so vehicles can be analyzed according to when they entered the fleet. Invalid date values become `NaT` and are recorded.

The vehicle dataset is not reduced because of missing non-identifying vehicle information. Such values remain available for later investigation.

In [6]:
vehicles_clean = vehicles.copy()

# Remove extra spaces and standardize IDs
vehicles_clean["vehicle_id"] = vehicles_clean["vehicle_id"].str.strip().str.upper()

# Remove records without vehicle IDs
before = len(vehicles_clean)
vehicles_clean = vehicles_clean.dropna(subset=["vehicle_id"])
log_cleaning(
    "Vehicles",
    "Remove missing vehicle IDs",
    before,
    len(vehicles_clean),
    "A vehicle must have an ID to connect it to charging sessions."
)

# Remove duplicate vehicle IDs
before = len(vehicles_clean)
vehicles_clean = vehicles_clean.drop_duplicates(
    subset=["vehicle_id"],
    keep="first"
)
log_cleaning(
    "Vehicles",
    "Remove duplicate vehicle IDs",
    before,
    len(vehicles_clean),
    "Each vehicle should appear only once."
)

numeric_columns = [
    "battery_kwh",
    "rated_kwh_per_100mi",
    "replaced_gas_car_mpg"
]

for column in numeric_columns:

    before_invalid = vehicles_clean[column].isna().sum()

    original_values = vehicles_clean[column].copy()

    vehicles_clean[column] = pd.to_numeric(
        vehicles_clean[column],
        errors="coerce"
    )

    newly_invalid = (
        original_values.notna() &
        vehicles_clean[column].isna()
    ).sum()

    log_quality(
        "Vehicles",
        column,
        newly_invalid,
        "Converted invalid numerical values to NaN",
        "The column must contain numeric values for calculations."
    )

original_registration = vehicles_clean["registration_month"].copy()

vehicles_clean["registration_month"] = pd.to_datetime(
    vehicles_clean["registration_month"],
    format="%Y-%m",
    errors="coerce"
)

invalid_registration = (
    original_registration.notna() &
    vehicles_clean["registration_month"].isna()
).sum()

log_quality(
    "Vehicles",
    "registration_month",
    invalid_registration,
    "Converted invalid dates to NaT",
    "Registration dates are needed to identify when vehicles entered the fleet."
)

print("Clean vehicle records:", len(vehicles_clean))
print("Unique vehicle IDs:", vehicles_clean["vehicle_id"].nunique())

Clean vehicle records: 10000
Unique vehicle IDs: 10000


# 6. Clean the Public Charging Station Dataset

The public station dataset is prepared so that public charging sessions can be matched to known stations.

The following operations are performed:

1. **Standardize station IDs:** Remove extra spaces and convert station IDs to uppercase.
2. **Remove missing station IDs:** A station without an identifier cannot be matched to a charging session.
3. **Remove duplicate station IDs:** Each public station should appear only once. When duplicate IDs occur, the first record is retained.
4. **Convert opening dates:** The station opening date is converted to a datetime value. Invalid dates become `NaT`.

Home charging stations are not expected to appear in this dataset because it contains public stations only. Home charging stations are checked separately using the vehicle identifier.

In [7]:
stations_clean = stations.copy()

# Standardize station IDs
stations_clean["station_id"] = (
    stations_clean["station_id"].str.strip().str.upper()
)

# Remove missing station IDs
before = len(stations_clean)
stations_clean = stations_clean.dropna(subset=["station_id"])

log_cleaning(
    "Stations",
    "Remove missing station IDs",
    before,
    len(stations_clean),
    "A public station must have an ID to match charging sessions."
)

# Remove duplicate station IDs
before = len(stations_clean)
stations_clean = stations_clean.drop_duplicates(
    subset=["station_id"],
    keep="first"
)

log_cleaning(
    "Stations",
    "Remove duplicate station IDs",
    before,
    len(stations_clean),
    "Each public charging station should appear once."
)

# Convert station opening date
stations_clean["open_date"] = pd.to_datetime(
    stations_clean["open_date"],
    errors="coerce"
)

print("Clean public stations:", len(stations_clean))
print("Unique station IDs:", stations_clean["station_id"].nunique())

Clean public stations: 239
Unique station IDs: 239


# 7. Clean the Charging Sessions Dataset

The charging sessions dataset contains the individual charging events used in the project's analysis. It is therefore important to preserve valid sessions while identifying incomplete or inconsistent records.

The following operations are performed:

1. **Standardize text fields:** Remove leading and trailing spaces from identifiers and descriptive columns.
2. **Standardize identifiers:** Convert session, vehicle, and station IDs to uppercase.
3. **Standardize charging location:** Convert the home/public field to lowercase so that the two categories can be consistently identified.
4. **Standardize charger information:** Convert port types to lowercase and connector types to uppercase.
5. **Convert energy measurements:** Convert `kwh` to a numeric value. Values that cannot be converted become `NaN` and are recorded in the data-quality log.
6. **Convert timestamps:** Convert session start and end times into datetime values. Invalid timestamps become `NaT` and are recorded.

The charging timestamps are recorded as local clock times without an explicit UTC offset. This creates a potential daylight-saving-time ambiguity during the fall clock change, when the 1:00 AM hour occurs twice.

For this reason, duration validation is postponed until the daylight-saving-time issue can be investigated. This avoids incorrectly removing potentially valid sessions.

In [8]:
charging_clean = charging.copy()

# Standardize text columns
text_columns = [
    "session_id",
    "vehicle_id",
    "station_id",
    "port_type",
    "connector",
    "home_or_public"
]

for column in text_columns:
    charging_clean[column] = charging_clean[column].str.strip()

# Standardize identifiers
charging_clean["session_id"] = charging_clean["session_id"].str.upper()
charging_clean["vehicle_id"] = charging_clean["vehicle_id"].str.upper()
charging_clean["station_id"] = charging_clean["station_id"].str.upper()

# Standardize charging location
charging_clean["home_or_public"] = (
    charging_clean["home_or_public"].str.lower()
)

# Standardize charger types
charging_clean["port_type"] = (
    charging_clean["port_type"].str.lower()
)

charging_clean["connector"] = (
    charging_clean["connector"].str.upper()
)

original_kwh = charging_clean["kwh"].copy()

charging_clean["kwh"] = pd.to_numeric(
    charging_clean["kwh"],
    errors="coerce"
)

invalid_kwh = (
    original_kwh.notna() &
    charging_clean["kwh"].isna()
).sum()

log_quality(
    "Charging Sessions",
    "kwh",
    invalid_kwh,
    "Converted invalid numerical values to NaN",
    "Energy consumption must be numeric to calculate charging emissions."
)

# Convert start timestamps
original_start = charging_clean["start_time"].copy()

charging_clean["start_time"] = pd.to_datetime(
    charging_clean["start_time"],
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)

invalid_start = (
    original_start.notna() &
    charging_clean["start_time"].isna()
).sum()

log_quality(
    "Charging Sessions",
    "start_time",
    invalid_start,
    "Converted invalid timestamps to NaT",
    "Valid timestamps are required to determine when charging occurred."
)

# Convert end timestamps
original_end = charging_clean["end_time"].copy()

charging_clean["end_time"] = pd.to_datetime(
    charging_clean["end_time"],
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)

invalid_end = (
    original_end.notna() &
    charging_clean["end_time"].isna()
).sum()

log_quality(
    "Charging Sessions",
    "end_time",
    invalid_end,
    "Converted invalid timestamps to NaT",
    "Valid timestamps are required to calculate charging duration."
)

print("Charging sessions inspected:", len(charging_clean))

Charging sessions inspected: 2927187


# 7.1 Preserve and Remove Sessions with Missing Essential Information

Before removing incomplete charging sessions, records with missing energy measurements are saved separately as `sessions_missing_energy.csv`.

These records are preserved for investigation because missing energy measurements cannot be used to calculate charging emissions. They are not automatically assigned an energy value of zero.

The notebook then reports the number of missing values in each required column.

The following fields are considered essential:
- Session ID.
- Vehicle ID.
- Station ID.
- Start time.
- End time.
- Energy delivered (`kwh`).
- Charging location (`home_or_public`).

Records missing any of these required values are removed from the main analysis dataset and the number removed is recorded.

This step ensures that the remaining sessions have the minimum information necessary for matching, time-based analysis, and energy calculations.

In [9]:
required_columns = [
    "session_id",
    "vehicle_id",
    "station_id",
    "start_time",
    "end_time",
    "kwh",
    "home_or_public"
]

before = len(charging_clean)

missing_energy_sessions = charging_clean[
    charging_clean["kwh"].isna()
].copy()

missing_energy_sessions.to_csv(
    "sessions_missing_energy.csv",
    index=False
)

print(
    "Sessions saved for missing-energy investigation:",
    len(missing_energy_sessions)
)

print("MISSING REQUIRED VALUES BEFORE REMOVAL")
print("--------------------------------------")

for column in required_columns:
    print(f"{column}: {charging_clean[column].isna().sum():,}")
charging_clean = charging_clean.dropna(
    subset=required_columns
)

log_cleaning(
    "Charging Sessions",
    "Remove missing essential values",
    before,
    len(charging_clean),
    "Sessions need identifiers, timestamps, electricity consumption, and charging location."
)

Sessions saved for missing-energy investigation: 44045
MISSING REQUIRED VALUES BEFORE REMOVAL
--------------------------------------
session_id: 0
vehicle_id: 0
station_id: 0
start_time: 0
end_time: 0
kwh: 44,045
home_or_public: 0


# 7.2 Remove Duplicate Charging Sessions

Charging sessions are checked for duplicate session IDs.

Each session should represent one unique charging event. If the same session ID appears more than once, the notebook retains the first occurrence and removes subsequent occurrences.

This prevents the same charging event from being counted multiple times when calculating total energy consumption and emissions.

In [10]:
before = len(charging_clean)

charging_clean = charging_clean.drop_duplicates(
    subset=["session_id"],
    keep="first"
)

log_cleaning(
    "Charging Sessions",
    "Remove duplicate session IDs",
    before,
    len(charging_clean),
    "Each charging session should appear only once."
)

# 7.3 Remove Invalid Energy and Charging Location Values

Charging sessions are checked for two additional problems:

1. **Invalid energy:** Sessions must have a positive energy value (`kwh > 0`).
2. **Invalid charging location:** The location must be identified as either `home` or `public`.

Session duration is intentionally not validated at this stage. Because the timestamps do not contain timezone offsets, a session crossing the repeated daylight-saving-time hour could appear to have a negative duration when interpreted as ordinary local timestamps. Duration validation will be addressed separately during timestamp and daylight-saving-time analysis.

In [11]:
# Remove invalid energy and location values only.
# Duration validation will happen after DST ambiguity is investigated.

before = len(charging_clean)

charging_clean = charging_clean[
    (charging_clean["kwh"] > 0) &
    (charging_clean["home_or_public"].isin(["home", "public"]))
].copy()

log_cleaning(
    "Charging Sessions",
    "Remove invalid energy and location records",
    before,
    len(charging_clean),
    "Sessions must have positive energy and a recognized charging location. Duration validation is deferred until DST handling."
)

# 8. Match Charging Sessions to Vehicles

Each charging session must be associated with one of the vehicles in the vehicle dataset.

The vehicle IDs from the cleaned vehicle dataset are compared with the vehicle IDs in the cleaned charging sessions dataset.

Two groups are identified:

- **Matched sessions:** The vehicle ID exists in the vehicle dataset.
- **Unmatched sessions:** The vehicle ID does not exist in the vehicle dataset.

The number of matched and unmatched sessions is reported.

Sessions without a matching vehicle are removed from the main analysis dataset because they cannot be reliably associated with the EV fleet being studied. The number removed and the reason are recorded.

In [12]:
# Identify sessions with no matching vehicle
vehicle_ids = set(vehicles_clean["vehicle_id"])

charging_clean["vehicle_match"] = (
    charging_clean["vehicle_id"].isin(vehicle_ids)
)

unmatched_vehicles = charging_clean[
    ~charging_clean["vehicle_match"]
].copy()

print("Sessions with matching vehicles:",
      charging_clean["vehicle_match"].sum())

print("Sessions without matching vehicles:",
      len(unmatched_vehicles))

Sessions with matching vehicles: 2868765
Sessions without matching vehicles: 0


In [13]:
before = len(charging_clean)

charging_clean = charging_clean[
    charging_clean["vehicle_match"]
].copy()

charging_clean = charging_clean.drop(columns=["vehicle_match"])

log_cleaning(
    "Charging Sessions",
    "Remove sessions without matching vehicles",
    before,
    len(charging_clean),
    "Every analyzed charging session must belong to one of the 10,000 mandate vehicles."
)

# 9. Match Charging Sessions to Stations

Charging sessions are matched to stations using different rules for home and public charging.

**Public charging:**
- The session's station ID must exist in the cleaned public station dataset.

**Home charging:**
- Home station IDs follow the expected format `H-` followed by the vehicle ID.
- The session's station ID must match the expected home station ID for that vehicle.

Sessions without a valid station relationship are removed from the main analysis dataset. The number removed and the reason are recorded.


In [14]:
public_station_ids = set(stations_clean["station_id"])

# Check public station matches
charging_clean["station_match"] = np.where(
    charging_clean["home_or_public"] == "public",
    charging_clean["station_id"].isin(public_station_ids),

    # Home station must match its vehicle ID
    charging_clean["station_id"] ==
    "H-" + charging_clean["vehicle_id"]
)

# Identify unmatched stations
unmatched_stations = charging_clean[
    ~charging_clean["station_match"]
].copy()

print("Sessions with matching stations:",
      charging_clean["station_match"].sum())

print("Sessions without matching stations:",
      len(unmatched_stations))

Sessions with matching stations: 2868765
Sessions without matching stations: 0


In [15]:
before = len(charging_clean)

charging_clean = charging_clean[
    charging_clean["station_match"]
].copy()

charging_clean = charging_clean.drop(
    columns=["station_match"]
)

log_cleaning(
    "Charging Sessions",
    "Remove sessions without matching stations",
    before,
    len(charging_clean),
    "Public sessions must match stations.csv; home sessions must match their vehicle's H- station ID."
)

# 10. Check Charging Energy Against Vehicle Battery Capacity

The cleaned vehicle dataset is merged with the charging sessions dataset to add each vehicle's usable battery capacity.

Each session's delivered energy is then compared with the vehicle's battery capacity.

Sessions with delivered energy greater than the listed battery capacity are flagged as potential data-quality issues and saved to `battery_capacity_issues.csv`.

This data is not automatically removed. They are preserved for further investigation because an unusually large session may indicate an error, an unusual charging event, or another issue.

This check helps identify potentially unrealistic energy measurements before calculating total EV charging energy and emissions.

In [25]:
battery_issues = charging_clean[
    charging_clean["kwh"] > charging_clean["battery_kwh"]
].copy()

print("Sessions exceeding battery capacity:", len(battery_issues))

Sessions exceeding battery capacity: 20


In [24]:
battery_issues.to_csv(
    "battery_capacity_issues.csv",
    index=False
)

# 11. Separate Home and Public Charging Sessions

The cleaned charging sessions are divided into two datasets:

- **Home charging:** Sessions identified as home charging.
- **Public charging:** Sessions identified as public charging.

The number of sessions in each group is reported.

The notebook also checks that the combined number of home and public sessions equals the total number of cleaned sessions.

This separation is important for the research questions, especially the investigation of overnight home charging and its relationship to electricity demand and carbon emissions.

In [18]:
# Separate home charging
home_charging = charging_clean[
    charging_clean["home_or_public"] == "home"
].copy()

# Separate public charging
public_charging = charging_clean[
    charging_clean["home_or_public"] == "public"
].copy()

print("Home charging sessions:", len(home_charging))
print("Public charging sessions:", len(public_charging))

print("Total separated sessions:",
      len(home_charging) + len(public_charging))

print("Total cleaned sessions:", len(charging_clean))

Home charging sessions: 2447994
Public charging sessions: 420771
Total separated sessions: 2868765
Total cleaned sessions: 2868765


# 12. Integrate Vehicle and Station Information

Additional information is added to the home and public charging datasets using their corresponding identifiers.

For both datasets, vehicle information is merged using `vehicle_id`.

For public charging sessions, station information is also merged using `station_id`.

The merges use a many-to-one validation rule. This checks that multiple charging sessions can connect to one vehicle or station without creating unintended duplicate rows from the reference datasets.

The resulting datasets contain charging information along with relevant vehicle details. Public charging sessions also contain public station details.

This integration makes it possible to examine charging patterns by vehicle characteristics, charging location, and public station information.

In [19]:
# Add vehicle information to home sessions
home_charging = home_charging.merge(
    vehicles_clean,
    on="vehicle_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_vehicle")
)

# Add vehicle and public station information
public_charging = public_charging.merge(
    vehicles_clean,
    on="vehicle_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_vehicle")
)

public_charging = public_charging.merge(
    stations_clean,
    on="station_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_station")
)

print("Home charging rows:", len(home_charging))
print("Public charging rows:", len(public_charging))

Home charging rows: 2447994
Public charging rows: 420771


# 13. Save the Cleaned Datasets

The following files are created:

- `charging_sessions_clean.csv`: All cleaned and matched charging sessions.
- `home_charging_clean.csv`: Cleaned home charging sessions with vehicle information.
- `public_charging_clean.csv`: Cleaned public charging sessions with vehicle and station information.
- `vehicles_clean.csv`: Cleaned vehicle dataset.
- `stations_clean.csv`: Cleaned public station dataset.

Additional exception files are also saved:
- `sessions_missing_energy.csv`: Sessions with missing energy measurements.
- `battery_capacity_issues.csv`: Sessions with energy greater than the listed vehicle battery capacity.

These exception files preserve information that may be useful when investigating unusual patterns in the data.

In [20]:
charging_clean.to_csv("charging_sessions_clean.csv", index=False)

home_charging.to_csv("home_charging_clean.csv", index=False)

public_charging.to_csv("public_charging_clean.csv", index=False)

vehicles_clean.to_csv("vehicles_clean.csv", index=False)

stations_clean.to_csv("stations_clean.csv", index=False)

print("All cleaned datasets saved!")

All cleaned datasets saved!


# 14. Review Cleaning Results and Final Row Counts

The final section summarizes the results of the cleaning process.

The cleaning log reports the number of records removed during each operation and the reason for each removal.

The data-quality log reports invalid conversions and other affected values that did not necessarily result in row removal.

The notebook also compares the original and final row counts for each dataset and calculates the percentage of rows retained.

Finally, missing values in the cleaned datasets are reported.


In [21]:
cleaning_report = show_cleaning_log()

display(cleaning_report)

,Dataset,Cleaning Step,Rows Before,Rows After,Rows Removed,Reason
0,Vehicles,Remove missing vehicle IDs,10000,10000,0,A vehicle must have an ID to connect it to cha...
1,Vehicles,Remove duplicate vehicle IDs,10000,10000,0,Each vehicle should appear only once.
2,Stations,Remove missing station IDs,239,239,0,A public station must have an ID to match char...
3,Stations,Remove duplicate station IDs,239,239,0,Each public charging station should appear once.
4,Charging Sessions,Remove missing essential values,2927187,2883142,44045,"Sessions need identifiers, timestamps, electri..."
5,Charging Sessions,Remove duplicate session IDs,2883142,2868795,14347,Each charging session should appear only once.
6,Charging Sessions,Remove invalid energy and location records,2868795,2868765,30,Sessions must have positive energy and a recog...
7,Charging Sessions,Remove sessions without matching vehicles,2868765,2868765,0,Every analyzed charging session must belong to...
8,Charging Sessions,Remove sessions without matching stations,2868765,2868765,0,Public sessions must match stations.csv; home ...


Final before and after report

In [22]:
# FINAL DATASET ROW COUNT COMPARISON

final_counts = {
    "Charging Sessions": len(charging_clean),
    "Vehicles": len(vehicles_clean),
    "Public Stations": len(stations_clean)
}

comparison = pd.DataFrame({
    "Dataset": original_counts.keys(),
    "Original Rows": original_counts.values(),
    "Final Rows": final_counts.values()
})

comparison["Rows Removed"] = (
    comparison["Original Rows"] - comparison["Final Rows"]
)

comparison["Percent Retained"] = (
    comparison["Final Rows"] /
    comparison["Original Rows"] * 100
).round(2)

display(comparison)

,Dataset,Original Rows,Final Rows,Rows Removed,Percent Retained
0,Charging Sessions,2927187,2868765,58422,98.0
1,Vehicles,10000,10000,0,100.0
2,Public Stations,239,239,0,100.0
